In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import iirnotch, filtfilt

FS = 250
CHANNELS = ["FZ", "C3", "CZ", "C4"]

DATA_DIR = Path("../../data")
PROCESSED_DIR = DATA_DIR / "Processed"

CLEAN_MANIFEST_PATH = PROCESSED_DIR / "clean_manifest.csv"

FINAL_DIR = PROCESSED_DIR / "final_clean_trials"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MANIFEST_PATH = PROCESSED_DIR / "final_clean_manifest.csv"

## First - loading the clean manifest

In [2]:
from pathlib import Path
clean_manifest = pd.read_csv(CLEAN_MANIFEST_PATH)

FILTERED_DIR = Path("../../data/Processed/filtered_trials")

clean_manifest["file_path"] = clean_manifest["trial_id"].apply(
    lambda trial_id: str(
        FILTERED_DIR / f"cellula_MI_data_{int(trial_id)}.csv"
    )
)

display(clean_manifest.head())


,trial_id,file_path,label,rejected
0,1,..\..\data\Processed\filtered_trials\cellula_M...,left,False
1,2,..\..\data\Processed\filtered_trials\cellula_M...,right,False
2,3,..\..\data\Processed\filtered_trials\cellula_M...,left,False
3,4,..\..\data\Processed\filtered_trials\cellula_M...,left,False
4,5,..\..\data\Processed\filtered_trials\cellula_M...,left,False


### Hashing each trial channels --> to check dublicates again

In [3]:
def signal_hash(file_path):
    df = pd.read_csv(file_path)

    signal = df[CHANNELS]

    hashed_values = pd.util.hash_pandas_object(
        signal,
        index=False
    ).values

    return hashlib.md5(
        hashed_values.tobytes()
    ).hexdigest()


clean_manifest["signal_hash"] = clean_manifest["file_path"].apply(
    signal_hash
)

print("Hashes calculated successfully.")

Hashes calculated successfully.


In [4]:
clean_manifest.head()

,trial_id,file_path,label,rejected,signal_hash
0,1,..\..\data\Processed\filtered_trials\cellula_M...,left,False,2e130d2459af5b3fd55404a7b8c09f1f
1,2,..\..\data\Processed\filtered_trials\cellula_M...,right,False,42bd9d92671419d38d8234875cf5f43f
2,3,..\..\data\Processed\filtered_trials\cellula_M...,left,False,b7aa966139582895be0e242cb05b2c03
3,4,..\..\data\Processed\filtered_trials\cellula_M...,left,False,208043011519395e9ada1bcbe4822612
4,5,..\..\data\Processed\filtered_trials\cellula_M...,left,False,904f55cb6d9240e5d976496eb23ae157


### checking dublicates channels 

In [5]:
duplicate_mask = clean_manifest["signal_hash"].duplicated(
    keep=False
)

duplicates = (
    clean_manifest[duplicate_mask]
    .sort_values("signal_hash")
    .copy()
)

print("Trials involved in duplicate groups:", len(duplicates))
print(
    "Number of duplicate groups:",
    duplicates["signal_hash"].nunique()
)

display(
    duplicates[
        ["trial_id", "label", "signal_hash"]
    ]
)

Trials involved in duplicate groups: 72
Number of duplicate groups: 36


,trial_id,label,signal_hash
924,1078,right,02beb510a57594574fddbd3a2f3bcd73
42,43,left,02beb510a57594574fddbd3a2f3bcd73
923,1077,right,02fb9f42f11477f2efbd062ca2e3ea68
41,42,left,02fb9f42f11477f2efbd062ca2e3ea68
921,1075,right,0b8b3ed1e623ba4cc85f6f536a08ba0a
...,...,...,...
1791,1957,left,cb3f4eb9813dbb8262f9703de5952a0e
37,38,left,cfa90b532a45dbcb3b302eb7c868211b
919,1073,left,cfa90b532a45dbcb3b302eb7c868211b
1162,1316,left,e33c7ab40a6b7f034dd1c4aec7297022


### checking dublicates channels with contradicting labels 

In [6]:
label_counts = (
    duplicates
    .groupby("signal_hash")["label"]
    .nunique()
)

conflicting_hashes = label_counts[
    label_counts > 1
].index

conflicting_duplicates = duplicates[
    duplicates["signal_hash"].isin(conflicting_hashes)
].copy()

print(
    "Conflicting duplicate groups:",
    len(conflicting_hashes)
)

print(
    "Trials to remove:",
    len(conflicting_duplicates)
)

display(
    conflicting_duplicates[
        ["trial_id", "label", "signal_hash"]
    ]
)

Conflicting duplicate groups: 24
Trials to remove: 48


,trial_id,label,signal_hash
924,1078,right,02beb510a57594574fddbd3a2f3bcd73
42,43,left,02beb510a57594574fddbd3a2f3bcd73
923,1077,right,02fb9f42f11477f2efbd062ca2e3ea68
41,42,left,02fb9f42f11477f2efbd062ca2e3ea68
921,1075,right,0b8b3ed1e623ba4cc85f6f536a08ba0a
39,40,left,0b8b3ed1e623ba4cc85f6f536a08ba0a
1163,1317,right,1cd191600eed03e02c4c8cd9a3b1b115
515,669,left,1cd191600eed03e02c4c8cd9a3b1b115
771,925,left,32b9bc5b485210280d4414bc9c557a9a
528,682,right,32b9bc5b485210280d4414bc9c557a9a


### Removing contradicting ones 

In [7]:
conflicting_ids = set(
    conflicting_duplicates["trial_id"]
)

clean_v2_manifest = clean_manifest[
    ~clean_manifest["trial_id"].isin(conflicting_ids)
].copy()

clean_v2_manifest.reset_index(
    drop=True,
    inplace=True
)

print("Before:", len(clean_manifest))
print("Removed:", len(conflicting_ids))
print("Remaining:", len(clean_v2_manifest))

print()
print(clean_v2_manifest["label"].value_counts())

Before: 1995
Removed: 48
Remaining: 1947

label
right    975
left     972
Name: count, dtype: int64


In [8]:
duplicate_report_path = (
    PROCESSED_DIR / "conflicting_duplicates.csv"
)

conflicting_duplicates[
    ["trial_id", "label", "signal_hash"]
].to_csv(
    duplicate_report_path,
    index=False
)

print("Saved:", duplicate_report_path)

Saved: ..\..\data\Processed\conflicting_duplicates.csv


### for One label dublicates , keep one of them 

In [9]:
clean_v2_manifest = (
    clean_v2_manifest
    .drop_duplicates(
        subset="signal_hash",
        keep="first"
    )
    .reset_index(drop=True)
)

print("Original clean trials:", len(clean_manifest))
print("Final unique clean trials:", len(clean_v2_manifest))

print()
print(clean_v2_manifest["label"].value_counts())

Original clean trials: 1995
Final unique clean trials: 1935

label
right    971
left     964
Name: count, dtype: int64


### Making sure v2 is pure 

In [10]:
duplicate_mask = clean_v2_manifest["signal_hash"].duplicated(
    keep=False
)

duplicates = (
    clean_v2_manifest[duplicate_mask]
    .sort_values("signal_hash")
    .copy()
)

print("Trials involved in duplicate groups:", len(duplicates))
print(
    "Number of duplicate groups:",
    duplicates["signal_hash"].nunique()
)

display(
    duplicates[
        ["trial_id", "label", "signal_hash"]
    ]
)

Trials involved in duplicate groups: 0
Number of duplicate groups: 0


,trial_id,label,signal_hash


## Applying Notch filter 

In [11]:
NOTCH_FREQ = 50
Q = 30

b_notch, a_notch = iirnotch(
    NOTCH_FREQ,
    Q,
    fs=FS
)


def apply_notch(df):
    df = df.copy()

    for channel in CHANNELS:
        df[channel] = filtfilt(
            b_notch,
            a_notch,
            df[channel].values
        )

    return df

In [12]:
final_records = []

for _, row in clean_v2_manifest.iterrows():

    df = pd.read_csv(row["file_path"])

    final_df = apply_notch(df)

    output_path = (
        FINAL_DIR /
        f"cellula_MI_data_{int(row['trial_id'])}.csv"
    )

    final_df.to_csv(
        output_path,
        index=False
    )

    final_records.append({
        "trial_id": row["trial_id"],
        "file_path": str(output_path),
        "label": row["label"]
    })


final_manifest = pd.DataFrame(final_records)

print("Final trials:", len(final_manifest))
display(final_manifest.head())

Final trials: 1935


,trial_id,file_path,label
0,1,..\..\data\Processed\final_clean_trials\cellul...,left
1,2,..\..\data\Processed\final_clean_trials\cellul...,right
2,3,..\..\data\Processed\final_clean_trials\cellul...,left
3,4,..\..\data\Processed\final_clean_trials\cellul...,left
4,5,..\..\data\Processed\final_clean_trials\cellul...,left


In [13]:
final_manifest.to_csv(
    FINAL_MANIFEST_PATH,
    index=False
)

print("Saved final manifest:")
print(FINAL_MANIFEST_PATH)

print()
print("Final class distribution:")
print(final_manifest["label"].value_counts())

Saved final manifest:
..\..\data\Processed\final_clean_manifest.csv

Final class distribution:
label
right    971
left     964
Name: count, dtype: int64
